<a href="https://colab.research.google.com/github/SiddhantUke/Standard-LLM/blob/main/PGSQL_Mini_Shop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGSQL Mini Shop — Real PostgreSQL in Google Colab

A mini shop database in real PostgreSQL — `customers`, `products`, `orders`, `accounts` — then CRUD, JOIN, GROUP BY, an ACID money transfer, and an index with `EXPLAIN ANALYZE`.

No API key, no signup, no Docker, no local install. Colab gives you a fresh Linux machine — we install real PostgreSQL on it.




---



## Step 1 — Install real PostgreSQL in Colab

This cell installs the real PostgreSQL server on this Colab machine and sets the `postgres` user's password to `postgres`.

Takes about 2 minutes. Wait for it to finish — do not run the next cell early.

**Expected output:** `Postgres installed and running`


In [7]:
!apt-get update -qq
!apt-get install -y -qq postgresql postgresql-contrib > /dev/null
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
print("Postgres installed and running")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]
ALTER ROLE
Postgres installed and running




---



## Step 2 — Install the Python driver and connect

This cell installs `psycopg`, the Python driver for PostgreSQL, and connects Python to the server you just installed.

`conn` is the connection, `cur` is the cursor that runs SQL. The password here must match the one set in Step 1.

**Expected output:** a version line starting `PostgreSQL 16...`, then `Connected to Postgres`


In [8]:
!pip install -q "psycopg[binary]"
import psycopg
conn = psycopg.connect("host=localhost dbname=postgres user=postgres password=postgres", autocommit=True)
cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone()[0])
print("Connected to Postgres")


PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
Connected to Postgres




---



## Step 3 — Create the database and tables

This cell creates a fresh database called `shop_db` and builds 4 tables inside it: `customers`, `products`, `orders`, `accounts`.

The rules are built into the schema itself: `CHECK (balance >= 0)` stops a negative balance, `FOREIGN KEY` stops an order for a customer who does not exist, `UNIQUE` stops a duplicate email.

**Expected output:** `Tables created: customers, products, orders, accounts`


In [9]:
# If shop_db already exists from an earlier run, delete it first
cur.execute("DROP DATABASE IF EXISTS shop_db WITH (FORCE);")


# Create a fresh, empty database called shop_db
cur.execute("CREATE DATABASE shop_db;")


# Close the old connection, which was connected to the default postgres database
conn.close()


# Reconnect, this time to shop_db, with autocommit as a keyword argument
conn = psycopg.connect("host=localhost dbname=shop_db user=postgres password=postgres", autocommit=True)


# Make a new cursor on the new connection
cur = conn.cursor()


# Send all 4 CREATE TABLE statements in one go

cur.execute('''
-- Table 1: customers, one row per customer


CREATE TABLE customers (
  -- id auto-increases (1, 2, 3...) and is the primary key
  id SERIAL PRIMARY KEY,
  -- name can never be empty
  name TEXT NOT NULL,
  -- city can never be empty
  city TEXT NOT NULL,
  -- email must be unique, no two customers can share one
  email TEXT UNIQUE NOT NULL
);


-- Table 2: products, one row per product


CREATE TABLE products (
  -- id auto-increases and is the primary key
  id SERIAL PRIMARY KEY,
  -- product name can never be empty
  name TEXT NOT NULL,
  -- price is money with 2 decimals, and must be above 0
  price NUMERIC(10,2) NOT NULL CHECK (price > 0),
  -- stock can never go below 0
  stock INT NOT NULL CHECK (stock >= 0)
);



-- Table 3: orders, one row per order


CREATE TABLE orders (
  -- id auto-increases and is the primary key
  id SERIAL PRIMARY KEY,
  -- foreign key: the order must belong to a customer who exists
  customer_id INT REFERENCES customers(id),
  -- order total, can never be empty
  total NUMERIC(10,2) NOT NULL,
  -- order time fills itself automatically with the current time
  ordered_at TIMESTAMP DEFAULT NOW()
);



-- Table 4: accounts, the bank table for the transfer test later


CREATE TABLE accounts (
  -- id auto-increases and is the primary key
  id SERIAL PRIMARY KEY,
  -- account holder name can never be empty
  name TEXT NOT NULL,
  -- balance can never go below 0, this is what makes the bad transfer fail
  balance NUMERIC(10,2) NOT NULL CHECK (balance >= 0)
);
''')



# Confirmation line, so you know all 4 tables were created
print("Tables created: customers, products, orders, accounts")


Tables created: customers, products, orders, accounts




---



## Step 4 — Seed data

This cell fills the empty tables with sample data so you have something real to query: 10 customers, 8 products, 6 orders, 2 bank accounts.

You do not give an `id` — `SERIAL` fills it automatically, 1 to 10, in the order listed. Siddhant is customer 1, Rahul is 2.

**Expected output:**


In [10]:
# Insert all the sample data in one go: customers, products, orders, accounts
cur.execute('''
-- Insert 10 customers, ids will be 1 to 10 in this order
INSERT INTO customers (name, city, email) VALUES
 ('Siddhant Uke','Gurgaon','siddhant@example.com'),
 ('Rahul Verma','Delhi','rahul@example.com'),
 ('Palak Sharma','Amravati','palak@example.com'),
 ('Aarav Mehta','Mumbai','aarav@example.com'),
 ('Ishita Rao','Bengaluru','ishita@example.com'),
 ('Kabir Singh','Pune','kabir@example.com'),
 ('Ananya Iyer','Chennai','ananya@example.com'),
 ('Rohan Gupta','Noida','rohan@example.com'),
 ('Sneha Patil','Nagpur','sneha@example.com'),
 ('Vikram Yadav','Jaipur','vikram@example.com');
-- Insert 8 products with price and stock
INSERT INTO products (name, price, stock) VALUES
 ('Laptop', 55000, 10), ('Mouse', 799, 100), ('Keyboard', 1999, 50),
 ('Monitor', 12000, 20), ('Headphones', 2999, 40), ('Webcam', 3499, 25),
 ('SSD 512GB', 4500, 30), ('USB-C Hub', 1499, 60);
-- Insert 6 orders, customer_id points to the customers above
INSERT INTO orders (customer_id, total) VALUES
 (1, 55799), (1, 1999), (2, 12000), (3, 2999), (4, 55000), (5, 1499);
-- Insert 2 bank accounts for the transfer test later
INSERT INTO accounts (name, balance) VALUES
 ('Siddhant', 50000), ('Rahul', 20000);
''')
# Count the customers to check the insert worked
cur.execute("SELECT COUNT(*) FROM customers;")
# Print the customer count
print("customers:", cur.fetchone()[0])
# Count the products to check the insert worked
cur.execute("SELECT COUNT(*) FROM products;")
# Print the product count
print("products:", cur.fetchone()[0])
# Count the orders to check the insert worked
cur.execute("SELECT COUNT(*) FROM orders;")
# Print the order count
print("orders:", cur.fetchone()[0])


customers: 10
products: 8
orders: 6




---



## Step 5 — CRUD: read, update, delete

CRUD means Create, Read, Update, Delete. Create was Step 4 (INSERT). This cell does the other three.

Read products under 5000, update the Mouse stock after a sale, delete one test order. Never run UPDATE or DELETE without WHERE — without it, every row changes.

**Expected output:** 6 products under 5000, `Mouse after sale: ('Mouse', 95)`, `orders after delete: 5`


In [11]:
# READ: get name and price of all products under 5000, cheapest first
cur.execute("SELECT name, price FROM products WHERE price < 5000 ORDER BY price;")


# Loop over every row the query returned, one by one
for row in cur.fetchall():
    # Print one product row, like ('Mouse', 799.00)
    print(row)



# Print a divider line so the three parts are easy to read
print("---")


# UPDATE: a Mouse was just sold, so reduce its stock by 5 (100 becomes 95)
cur.execute("UPDATE products SET stock = stock - 5 WHERE name = 'Mouse';")

# Read the Mouse row back to check the update worked
cur.execute("SELECT name, stock FROM products WHERE name = 'Mouse';")

# Print the Mouse row after the sale
print("Mouse after sale:", cur.fetchone())

# Print a divider line so the three parts are easy to read
print("---")

# DELETE: remove order number 6, the last test order

cur.execute("DELETE FROM orders WHERE id = 6;")

# Count the orders left to check the delete worked
cur.execute("SELECT COUNT(*) FROM orders;")

# Print the order count after the delete
print("orders after delete:", cur.fetchone()[0])


('Mouse', Decimal('799.00'))
('USB-C Hub', Decimal('1499.00'))
('Keyboard', Decimal('1999.00'))
('Headphones', Decimal('2999.00'))
('Webcam', Decimal('3499.00'))
('SSD 512GB', Decimal('4500.00'))
---
Mouse after sale: ('Mouse', 95)
---
orders after delete: 5




---



## Step 6 — JOIN + GROUP BY: who spent the most?

This is the query interviewers love. JOIN matches each order to its customer, GROUP BY totals the spend per customer, ORDER BY puts the biggest spender on top.

**Expected first row:** `('Siddhant Uke', 'Gurgaon', 2, 57798.00)` — Laptop + Mouse (55799) and Keyboard (1999).


In [12]:
# Find the top customers: join customers with their orders, then total up the spend per customer

cur.execute('''
-- Pick the columns we want: name, city, how many orders, and total money spent
SELECT c.name, c.city,
COUNT(o.id) AS orders,
SUM(o.total) AS spent

-- Start from the customers table, call it c for short
FROM customers c

-- JOIN the orders table, call it o, matching each order to its customer
JOIN orders o
ON o.customer_id = c.id

-- Group the joined rows by customer, so COUNT and SUM work per person, not over everyone
GROUP BY c.name, c.city

-- Sort by total spent, biggest first, so the top customer is on top
ORDER BY spent DESC;

''')

# Loop over every row the query returned, one by one
for row in cur.fetchall():
    # Print one customer row, like ('Siddhant Uke', 'Gurgaon', 2, 57798.00)
    print(row)


('Siddhant Uke', 'Gurgaon', 2, Decimal('57798.00'))
('Aarav Mehta', 'Mumbai', 1, Decimal('55000.00'))
('Rahul Verma', 'Delhi', 1, Decimal('12000.00'))
('Palak Sharma', 'Amravati', 1, Decimal('2999.00'))




---



## Step 7 — ACID transfer: the bank test

Transfer 10000 from Siddhant to Rahul. Both updates must happen together, or neither happens.

This connection has NO autocommit on purpose — both updates form one transaction. The bad transfer of 999999 breaks the `CHECK (balance >= 0)` rule, so the whole transaction rolls back and the balances stay unchanged.

**Expected output:** after the good transfer Siddhant 40000, Rahul 30000. After the bad transfer, still 40000 / 30000.


In [13]:
# Define a reusable function: transfer money from one account to another

def transfer(from_id, to_id, amount):
    # Try the transfer, and if anything fails, jump to the except block below
    try:
        # Open a NEW connection with NO autocommit, so both updates form one transaction
        with psycopg.connect("host=localhost dbname=shop_db user=postgres password=postgres") as c:
            # Open a cursor on this connection, it closes itself when the block ends
            with c.cursor() as cu:
                # Step 1 of the transfer: subtract the amount from the sender
                cu.execute("UPDATE accounts SET balance = balance - %s WHERE id = %s", (amount, from_id))
                # Step 2 of the transfer: add the amount to the receiver
                cu.execute("UPDATE accounts SET balance = balance + %s WHERE id = %s", (amount, to_id))
            # Both updates worked, so save them permanently
            c.commit()
        # Tell us the transfer succeeded
        print(f"Transfer OK: {amount}")
    # If ANY line above failed, come here instead
    except Exception as e:
        # Print the error, the transaction has already rolled back, nothing was saved
        print("Transfer FAILED, rolled back:", e)

# Define a small helper: print both account balances, in id order
def show_balances():
    # Get name and balance of every account, Siddhant first, Rahul second
    cur.execute("SELECT name, balance FROM accounts ORDER BY id;")
    # Loop over the rows and print each one
    for row in cur.fetchall():
        # Print one account row, like ('Siddhant', 50000.00)
        print(row)

# Print the balances before any transfer
print("Before:"); show_balances()
# Good transfer: move 10000 from account 1 (Siddhant) to account 2 (Rahul)
transfer(1, 2, 10000)
# Print the balances after the good transfer
print("After good transfer:"); show_balances()
# Bad transfer: try to move 999999, far more than Siddhant has
transfer(1, 2, 999999)
# Print the balances after the bad transfer, they must be unchanged
print("After bad transfer (must be unchanged):"); show_balances()


Before:
('Siddhant', Decimal('50000.00'))
('Rahul', Decimal('20000.00'))
Transfer OK: 10000
After good transfer:
('Siddhant', Decimal('40000.00'))
('Rahul', Decimal('30000.00'))
Transfer FAILED, rolled back: new row for relation "accounts" violates check constraint "accounts_balance_check"
DETAIL:  Failing row contains (1, Siddhant, -959999.00).
After bad transfer (must be unchanged):
('Siddhant', Decimal('40000.00'))
('Rahul', Decimal('30000.00'))




---



## Step 8 — Index + EXPLAIN ANALYZE: make it fast

`EXPLAIN ANALYZE` shows HOW Postgres runs a query and how long it takes. First you see `Seq Scan` (reads the whole table). Then you add an index on `customer_id`.

On only 5 rows the planner STILL picks Seq Scan — reading 5 rows is cheaper than using the index. That is correct behaviour, not a bug. `SET enable_seqscan = OFF` only lets you see the `Index Scan` plan. At real scale (lakhs of rows) the planner picks the index on its own.

**Look for:** `Seq Scan`, then `Seq Scan` again, then `Index Scan using idx_orders_customer`


In [14]:
# Ask Postgres HOW it will run this query, and actually run it and time it
cur.execute("EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 1;")



# The answer comes back as many text lines, join them with newlines and print them
print("\n".join(r[0] for r in cur.fetchall()))


# Create an index on customer_id, like the index at the back of a book
cur.execute("CREATE INDEX idx_orders_customer ON orders(customer_id);")
# Print a divider, and warn that the planner will still pick Seq Scan on only 5 rows
print("--- after index (planner still picks Seq Scan on 5 rows: it is cheaper) ---")



# Ask Postgres HOW it will run the same query again, now that the index exists
cur.execute("EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 1;")
# Print the plan lines again
print("\n".join(r[0] for r in cur.fetchall()))
# Print a divider, this next part is only to SEE the index plan
print("--- same query with Seq Scan disabled, so you can see the index plan ---")


# Tell Postgres: for this session only, do not use Seq Scan
cur.execute("SET enable_seqscan = OFF;")

# Ask Postgres HOW it will run the same query one more time
cur.execute("EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 1;")

# Print the plan lines one last time, this time you should see Index Scan
print("\n".join(r[0] for r in cur.fetchall()))


Seq Scan on orders  (cost=0.00..27.00 rows=7 width=32) (actual time=0.010..0.012 rows=2 loops=1)
  Filter: (customer_id = 1)
  Rows Removed by Filter: 3
Planning Time: 0.059 ms
Execution Time: 0.025 ms
--- after index (planner still picks Seq Scan on 5 rows: it is cheaper) ---
Seq Scan on orders  (cost=0.00..1.06 rows=1 width=32) (actual time=0.005..0.006 rows=2 loops=1)
  Filter: (customer_id = 1)
  Rows Removed by Filter: 3
Planning Time: 0.127 ms
Execution Time: 0.014 ms
--- same query with Seq Scan disabled, so you can see the index plan ---
Index Scan using idx_orders_customer on orders  (cost=0.13..8.15 rows=1 width=32) (actual time=0.021..0.022 rows=2 loops=1)
  Index Cond: (customer_id = 1)
Planning Time: 0.035 ms
Execution Time: 0.029 ms


In [15]:
# Add a new customer from your own city, change these three values to yours
cur.execute("INSERT INTO customers (name, city, email) VALUES ('Siddhant Uke', 'Gurgaon', 'siddhant2@example.com');")

# Find the id Postgres just gave your new customer
cur.execute("SELECT id FROM customers WHERE email = 'siddhant2@example.com';")

# Print that id, you will need it for the next line
print("new customer id:", cur.fetchone()[0])


new customer id: 11


In [18]:
# Place an order for your new customer for the Headphones, 2999
cur.execute("INSERT INTO orders (customer_id, total) VALUES (11, 2999);")

# Count the orders again to check it went in
cur.execute("SELECT COUNT(*) FROM orders;")

# Print the order count, it should now be 6 again
print("orders now:", cur.fetchone()[0])


orders now: 7


## Bonus — See the whole data

`SELECT *` means all columns, all rows. This cell prints every row of all 4 tables, so you can check your work at a glance.

**Check:** Mouse stock should be 95, orders should be 5, accounts should be Siddhant 40000 and Rahul 30000.


In [19]:
# Create a new table: one row per product inside an order
cur.execute('''
CREATE TABLE order_items (
  id SERIAL PRIMARY KEY,
  order_id INT REFERENCES orders(id),
  product_id INT REFERENCES products(id),
  qty INT NOT NULL CHECK (qty > 0)
);
''')
# Confirmation line
print("order_items table created")


order_items table created


In [20]:
# Try to transfer 50000 when Siddhant has only 40000, this must fail and roll back
transfer(1, 2, 50000)

# Print the balances after the failed transfer, they must be unchanged
print("After break-it transfer:"); show_balances()


Transfer FAILED, rolled back: new row for relation "accounts" violates check constraint "accounts_balance_check"
DETAIL:  Failing row contains (1, Siddhant, -10000.00).
After break-it transfer:
('Siddhant', Decimal('40000.00'))
('Rahul', Decimal('30000.00'))


In [28]:
# Print a heading so you know which table is coming next
print("=== customers ===")
print("-------------------------------------------------------------------------------------------------------")

# Get every column and every row from the customers table
cur.execute("SELECT * FROM customers;")
# Loop over all the rows and print each one
for row in cur.fetchall():

    # Print one customer row
    print(row)


=== customers ===
-------------------------------------------------------------------------------------------------------
(1, 'Siddhant Uke', 'Gurgaon', 'siddhant@example.com')
(2, 'Rahul Verma', 'Delhi', 'rahul@example.com')
(3, 'Palak Sharma', 'Amravati', 'palak@example.com')
(4, 'Aarav Mehta', 'Mumbai', 'aarav@example.com')
(5, 'Ishita Rao', 'Bengaluru', 'ishita@example.com')
(6, 'Kabir Singh', 'Pune', 'kabir@example.com')
(7, 'Ananya Iyer', 'Chennai', 'ananya@example.com')
(8, 'Rohan Gupta', 'Noida', 'rohan@example.com')
(9, 'Sneha Patil', 'Nagpur', 'sneha@example.com')
(10, 'Vikram Yadav', 'Jaipur', 'vikram@example.com')
(11, 'Siddhant Uke', 'Gurgaon', 'siddhant2@example.com')


In [29]:
# Print a heading for the next table
print("=== products ===")
print("-------------------------------------------------------------------------------------------------------")

# Get every column and every row from the products table
cur.execute("SELECT * FROM products;")
# Loop over all the rows and print each one
for row in cur.fetchall():
    # Print one product row
    print(row)

=== products ===
-------------------------------------------------------------------------------------------------------
(1, 'Laptop', Decimal('55000.00'), 10)
(3, 'Keyboard', Decimal('1999.00'), 50)
(4, 'Monitor', Decimal('12000.00'), 20)
(5, 'Headphones', Decimal('2999.00'), 40)
(6, 'Webcam', Decimal('3499.00'), 25)
(7, 'SSD 512GB', Decimal('4500.00'), 30)
(8, 'USB-C Hub', Decimal('1499.00'), 60)
(2, 'Mouse', Decimal('799.00'), 95)


In [30]:
# Print a heading for the next table
print("=== orders ===")
print("-------------------------------------------------------------------------------------------------------")
# Get every column and every row from the orders table
cur.execute("SELECT * FROM orders;")
# Loop over all the rows and print each one
for row in cur.fetchall():
    # Print one order row
    print(row)

=== orders ===
-------------------------------------------------------------------------------------------------------
(1, 1, Decimal('55799.00'), datetime.datetime(2026, 11, 18, 5, 28, 29, 250083))
(2, 1, Decimal('1999.00'), datetime.datetime(2026, 11, 18, 5, 28, 29, 250083))
(3, 2, Decimal('12000.00'), datetime.datetime(2026, 11, 18, 5, 28, 29, 250083))
(4, 3, Decimal('2999.00'), datetime.datetime(2026, 11, 18, 5, 28, 29, 250083))
(5, 4, Decimal('55000.00'), datetime.datetime(2026, 11, 18, 5, 28, 29, 250083))
(7, 11, Decimal('2999.00'), datetime.datetime(2026, 11, 18, 5, 31, 43, 832205))
(8, 11, Decimal('2999.00'), datetime.datetime(2026, 11, 18, 5, 42, 40, 701544))


In [31]:
# Print a heading for the next table
print("=== accounts ===")

print("-------------------------------------------------------------------------------------------------------")
# Get every column and every row from the accounts table
cur.execute("SELECT * FROM accounts;")
# Loop over all the rows and print each one
for row in cur.fetchall():
    # Print one account row
    print(row)

=== accounts ===
-------------------------------------------------------------------------------------------------------
(1, 'Siddhant', Decimal('40000.00'))
(2, 'Rahul', Decimal('30000.00'))
